# **Sesión 2:** Calibración de Cámara 📷⚙️

## 0. Preparación

In [1]:
import cv2
print("OpenCV should be 4.8.x Current version:", cv2.__version__)
from typing import List
import numpy as np
# import imageio --- IGNORE IF DEPRECATED---
import imageio.v3 as imageio
import copy
import os
import glob

OpenCV should be 4.8.x Current version: 4.8.0


## **Apartado A: Calibración de cámara** (derecha e izquierda)

En este apartado se realiza la calibración de dos cámaras de un sistema estereoscópico. Para ello se trabajará con las imágenes de las carpetas ``left`` y ``right``. En primer lugar se trabajará con la carpeta ``left``. Posteriormente, deberá repetir el proceso con las imágenes en la carpeta ``right``. Ambas carpetas contienen imágenes con las que se calibrarán las cámaras. En ellas aparece el patrón de calibración en diferentes posiciones y orientaciones. Estas imágenes serán los datos de entrada.

Los pasos que deberá seguir para calibrar una cámara son:

1. Defina y ejecute el método para cargar imágenes ``load_images()``.
2. Detecte las esquinas de los patrones usando ``cv2.findChessboardCorners()``. Refine las detecciones con ``cv2.cornerSubPix()``.
3. Compruebe que las detecciones son correctas dibujando los resultados con ``cv2.drawChessboardCorners()``.
4. Defina y ejecute el método ``get_chessboard_points(chessboard_shape, dx, dy)`` que proporcione las coordenadas 3D de las esquinas del patrón. El sistema de referencia utilizado deberá estar anclado al propio patrón.
5. Utilice ``cv2.calibrateCamera`` para obtener los parámetros de calibración para la cámara izquierda.

### **Tarea A.1:** Defina y ejecute el método para cargar imágenes ``load_images()``.

In [2]:
def load_images(filenames: List) -> List:
    return [cv2.imread(filename) for filename in filenames]

In [3]:
# Build a list containing the paths of all images from the left camera
imgs_path = sorted(glob.glob(os.path.join("..", "data", "left", "*.jpg")))
print(f"{len(imgs_path)} imágenes encontradas")
imgs = load_images(imgs_path)   

19 imágenes encontradas


### **Tarea A.2:** Detecte las esquinas de los patrones usando ``cv2.findChessboardCorners()``. Refine las detecciones con ``cv2.cornerSubPix()``.

In [4]:
# TODO Find corners with cv2.findChessboardCorners()
# Hint: you can use a list comprehension to process all images
pattern_size = (8, 6)  # esquinas interiores (columnas, filas)
corners = [cv2.findChessboardCorners(img, pattern_size) for img in imgs]

In [5]:
corners_copy = copy.deepcopy(corners)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.01)

# TODO To refine corner detections with cv2.cornerSubPix() you need to input grayscale images. Build a list containing grayscale images.
imgs_gray = [cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) for img in imgs]

# Refinamos solo las imágenes con detección correcta (cor[0] == True)
corners_refined = [
    cv2.cornerSubPix(gray, cor[1], (8, 6), (-1, -1), criteria) if cor[0] else []
    for gray, cor in zip(imgs_gray, corners_copy)]

print(f"{sum(cor[0] for cor in corners)}/{len(corners)} detecciones correctas")

19/19 detecciones correctas


### **Tarea A.3:** Compruebe que las detecciones son correctas dibujando los resultados con ``cv2.drawChessboardCorners()``

In [6]:
imgs_copy = copy.deepcopy(imgs)

In [7]:
# TODO Use cv2.drawChessboardCorners() to draw the cornes
imgs_copy = [
    cv2.drawChessboardCorners(img, pattern_size, cor_ref, cor[0])
    for img, cor_ref, cor in zip(imgs_copy, corners_refined, corners)
]

In [8]:
# TODO Show images and save when needed

def show_image(img, title="Image"):
    cv2.imshow(title, img)
    cv2.waitKey(0)  # espera a que pulses una tecla
    cv2.destroyAllWindows()
    
def write_image(img, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    cv2.imwrite(path, img)

### **Tarea A.4:** Defina y ejecute el método ``get_chessboard_points(chessboard_shape, dx, dy)``

In [9]:
# TODO Design the method. It should return a np.array with np.float32 elements
def get_chessboard_points(chessboard_shape, dx, dy):
    cols, rows = chessboard_shape  # (8, 6): 8 esquinas por fila, 6 filas
    return np.array(
        [[i * dx, j * dy, 0] for i in range(rows) for j in range(cols)],
        dtype=np.float32
    )

In [10]:
# TODO You need the points for every image, not just one (consider a list comprehension)
chessboard_points = [get_chessboard_points(pattern_size, 30, 30)
                     for cor in corners_refined if len(cor) > 0]

### **Tarea A.5:** Utilice ``cv2.calibrateCamera()`` para obtener los parámetros de calibración para la cámara izquierda

In [12]:
# Filter data and get only those with adequate detections
valid_corners = [cor for cor in corners_refined if len(cor) > 0]
# Convert list to numpy array with np.float32 elements
valid_corners = np.asarray(valid_corners, dtype=np.float32)

In [13]:
# TODO
rms, intrinsics, dist_coeffs, rvecs, tvecs = cv2.calibrateCamera(
    chessboard_points,         
    valid_corners,              
    imgs_gray[0].shape[::-1],
    None,                        
    None    
)

# Obtain extrinsics
extrinsics = list(map(lambda rvec, tvec: np.hstack((cv2.Rodrigues(rvec)[0], tvec)), rvecs, tvecs))

In [14]:
# Print outputs
print("Intrinsics:\n", intrinsics)
print("Distortion coefficients:\n", dist_coeffs)
print("Root mean squared reprojection error:\n", rms)

Intrinsics:
 [[424.49980412   0.         155.58449296]
 [  0.         426.73945888 129.55971706]
 [  0.           0.           1.        ]]
Distortion coefficients:
 [[-5.27421036e-02 -1.46215131e+00  4.83483444e-03 -5.90090044e-03
   6.02537794e+00]]
Root mean squared reprojection error:
 0.1847854065061554


### **Pregunta A.1:** Repita el proceso (carga de imágenes, detección y comprobación de esquinas, etc.) para la cámara derecha.

In [15]:
# 1. Carga de imágenes de la cámara derecha
imgs_path_r = sorted(glob.glob(os.path.join("..", "data", "right", "*.jpg")))
imgs_r = load_images(imgs_path_r)

# 2. Detección de esquinas y refinamiento
corners_r = [cv2.findChessboardCorners(img, pattern_size) for img in imgs_r]
corners_copy_r = copy.deepcopy(corners_r)
imgs_gray_r = [cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) for img in imgs_r]
corners_refined_r = [
    cv2.cornerSubPix(gray, cor[1], (8, 6), (-1, -1), criteria) if cor[0] else []
    for gray, cor in zip(imgs_gray_r, corners_copy_r)]

print(f"{sum(cor[0] for cor in corners_r)}/{len(corners_r)} detecciones correctas")
print("Sin detección:", [os.path.basename(p) for p, cor in zip(imgs_path_r, corners_r) if not cor[0]])

# 3. Comprobación: dibujamos las esquinas y guardamos las imágenes
imgs_copy_r = copy.deepcopy(imgs_r)
imgs_copy_r = [
    cv2.drawChessboardCorners(img, pattern_size, cor_ref, cor[0]) if cor[0] else img
    for img, cor_ref, cor in zip(imgs_copy_r, corners_refined_r, corners_r)]
for path, img in zip(imgs_path_r, imgs_copy_r):
    write_image(img, os.path.join("..", "results", "right", os.path.basename(path)))

# 4. Puntos 3D del patrón (solo para imágenes con detección correcta)
valid_corners_r = [cor for cor in corners_refined_r if len(cor) > 0]
chessboard_points_r = [get_chessboard_points(pattern_size, 30, 30) for _ in valid_corners_r]
valid_corners_r = np.asarray(valid_corners_r, dtype=np.float32)

# 5. Calibración
rms_r, intrinsics_r, dist_coeffs_r, rvecs_r, tvecs_r = cv2.calibrateCamera(
    chessboard_points_r, valid_corners_r, imgs_gray_r[0].shape[::-1], None, None)
extrinsics_r = list(map(lambda rvec, tvec: np.hstack((cv2.Rodrigues(rvec)[0], tvec)), rvecs_r, tvecs_r))

print("Intrinsics:\n", intrinsics_r)
print("Distortion coefficients:\n", dist_coeffs_r)
print("Root mean squared reprojection error:\n", rms_r)

17/19 detecciones correctas
Sin detección: ['right_001.jpg', 'right_014.jpg']
Intrinsics:
 [[429.74668585   0.         146.02757684]
 [  0.         430.92875981 137.87387707]
 [  0.           0.           1.        ]]
Distortion coefficients:
 [[-0.11260577 -0.32047562  0.00637188 -0.00524419  2.19809693]]
Root mean squared reprojection error:
 0.1028161909271028


### **Pregunta A.2:** Diferencias entre cv2.findChessboardCorners() y cv2.cornerSubPix()

**``cv2.findChessboardCorners()``: detección global del patrón**

- **Entrada:** la imagen completa (color o gris) y el número de esquinas interiores del patrón (``pattern_size = (8, 6)``).
- **Qué hace:** binariza la imagen, localiza los cuadrados negros como cuadriláteros y comprueba que forman una rejilla del tamaño indicado. Las esquinas se obtienen de los puntos donde se tocan los cuadrados.
- **Salida:** un booleano ``ret`` que indica si se ha encontrado el patrón **completo** y las coordenadas de las esquinas **ordenadas** fila a fila (necesario para emparejarlas con los puntos 3D).
- **Precisión:** del orden de **1 píxel**.

**``cv2.cornerSubPix()``: refinamiento local subpíxel**

- **Entrada:** la imagen en **escala de grises**, las esquinas ya detectadas, un tamaño de ventana de búsqueda y un criterio de parada.
- **Qué hace:** no detecta nada, solo **mejora** esquinas que ya recibe. Se basa en que, en la esquina real $q$, el gradiente de la imagen en cualquier punto $p$ de su vecindad es perpendicular al vector $q - p$, es decir $\nabla I(p)^T (q - p) = 0$. Resuelve este sistema por mínimos cuadrados dentro de la ventana y repite iterativamente hasta cumplir el criterio (``EPS`` o ``MAX_ITER``).
- **Salida:** las mismas esquinas con precisión **subpíxel** (décimas o centésimas de píxel).

**Resumen de diferencias**

| | ``findChessboardCorners`` | ``cornerSubPix`` |
|---|---|---|
| Tipo | Detector global | Refinador local |
| Necesita esquinas previas | No | Sí |
| Imagen de entrada | Color o gris | Solo gris (usa gradientes) |
| Sabe que es un tablero | Sí | No (vale para cualquier esquina) |
| Precisión | ~1 px | Subpíxel |

**Resultados experimentales** (celda siguiente):

| Cámara | Desplazamiento medio | Desplazamiento máx. | RMS sin refinar | RMS refinado |
|---|---|---|---|---|
| Izquierda | 0.14 px | 2.98 px | 0.1846 px | 0.1848 px |
| Derecha | 0.10 px | 1.14 px | 0.1457 px | 0.1028 px |

El refinamiento mueve las esquinas en promedio solo una décima de píxel, porque en OpenCV 4.x ``findChessboardCorners`` ya aplica internamente un refinamiento ligero. Aun así, en la cámara derecha reduce el error de reproyección un 30 %. En la izquierda la diferencia es despreciable, lo que indica que las detecciones iniciales ya eran buenas. Como el coste es mínimo y puede mejorar notablemente la calibración, conviene aplicarlo siempre antes de calibrar.

In [ ]:
# Comparación: esquinas de findChessboardCorners() vs refinadas con cornerSubPix()
def compare_refinement(name, corners, corners_refined, imgs_gray):
    raw = [cor[1] for cor in corners if cor[0]]
    ref = [cor for cor in corners_refined if len(cor) > 0]
    dist = np.concatenate([np.linalg.norm(r - c, axis=2).ravel() for r, c in zip(ref, raw)])
    points = [get_chessboard_points(pattern_size, 30, 30) for _ in raw]
    size = imgs_gray[0].shape[::-1]
    rms_raw = cv2.calibrateCamera(points, np.asarray(raw, dtype=np.float32), size, None, None)[0]
    rms_ref = cv2.calibrateCamera(points, np.asarray(ref, dtype=np.float32), size, None, None)[0]
    print(f"{name}: desplazamiento medio {dist.mean():.3f} px, máx {dist.max():.3f} px | "
          f"RMS sin refinar {rms_raw:.4f} px, refinado {rms_ref:.4f} px")

compare_refinement("Izquierda", corners, corners_refined, imgs_gray)
compare_refinement("Derecha", corners_r, corners_refined_r, imgs_gray_r)

### **Pregunta A.3:** Número mínimo de imágenes necesarias para calibrar.

In [ ]:
# TODO Homework

## **Apartado B: Corrección de distorsión** (ojo de pez)

En este apartado se trabajará en la corrección de la distorsión debido a lentes de ojo de pez. Primero se calibrará una cámara con este tipo de lente, utilizando las imágenes de la carpeta ``fisheye``. Posteriormente se utilizarán estos parámetros de calibración para corregir la distorsión de una de las imágenes de calibración.

Los pasos que deberá seguir para calibrar una cámara con distorsión de ojo de pez son:

1. Reutilice el método ``load_images()`` para cargar las imágenes de la carpeta ``fisheye``.
2. Detecte las equinas procesando las imágenes con los métodos ``cv2.findChessboardCorners()`` y ``cv2.cornerSubPix()``.
3. Reutilice la función ``get_chessboard_points()`` para obtener las coordenadas del tablero.
4. Defina los argumentos para la función de calibración.
5. Calibre con ``cv2.fisheye.calibrate()``

### **Tarea B.1:** Reutilice el método ``load_images()`` para cargar las imágenes de la carpeta ``fisheye``

In [9]:
# TODO Build a list containing the paths of all images from the fisheye camera and load images
fisheye_imgs_path = sorted(glob.glob(os.path.join("..", "data", "fisheye", "*.jpg")))
fisheye_imgs = load_images(fisheye_imgs_path)

### **Tarea B.2:** Detecte las equinas procesando las imágenes con los métodos ``cv2.findChessboardCorners()`` y ``cv2.cornerSubPix()``.

In [10]:
imgs_corners = []
# Parameters for cv2.cornerSubPix()
subpix_criteria = (cv2.TERM_CRITERIA_EPS+cv2.TERM_CRITERIA_MAX_ITER, 30, 0.1)

#TODO Complete the required parts of the loop
for img in fisheye_imgs:
    
    # TODO parse arguments to cv2.findChessboardCorners()
    ret, corners = cv2.findChessboardCorners(image=img, patternSize=(7,6))
    
    # TODO convert image to grayscale to use cv2.cornerSubPix()
    gray_img = cv2.cvtColor(src=img, code= cv2.COLOR_BGR2GRAY)

    # TODO append only those refined_corners with proper detections
    if ret:
        refined_corners = cv2.cornerSubPix(gray_img, corners, (3,3), (-1,-1), subpix_criteria)
        imgs_corners.append(refined_corners)

print(f"Detecciones válidas: {len(imgs_corners)}/{len(fisheye_imgs)}")

Detecciones válidas: 10/10


### **Tarea B.3:** Reutilice la función ``get_chessboard_points()`` para obtener las coordenadas del tablero

In [11]:
# TODO Define the chessboard dimensions and the lenght of the squares (in [mm])
chessboard_dims = (7, 6)
length = 30
# TODO You need the points for every image, not just one (consider a list comprehension)
# cv2.fisheye.calibrate() expects each array with shape (n, 1, 3)
fisheye_chessboard_points = [get_chessboard_points(chessboard_dims, length, length).reshape(-1, 1, 3) for _ in imgs_corners]

NameError: name 'get_chessboard_points' is not defined

### **Tarea B.4:** Defina los argumentos para la calibración

In [ ]:
# Parameters for cv2.fisheye.calibrate()
calibration_flags = cv2.fisheye.CALIB_RECOMPUTE_EXTRINSIC+cv2.fisheye.CALIB_FIX_SKEW
intrinsics = np.zeros((3, 3))
distortion = np.zeros((4, 1))
rotations = [np.zeros((1, 1, 3), dtype=np.float64) for _ in imgs_corners]
traslations = [np.zeros((1, 1, 3), dtype=np.float64) for _ in imgs_corners]


### **Tarea B.5:** Calibración

In [ ]:
rms, intrinsics, distortion, rotations, traslations = cv2.fisheye.calibrate(fisheye_chessboard_points, imgs_corners, gray_img.shape[::-1], intrinsics, distortion, rotations, traslations, calibration_flags, subpix_criteria)

In [ ]:
# Show intrinsic matrix and distortion coefficients values
print(intrinsics)
print(distortion)
print(rms)

### **Pregunta B.1:** Corrija la distorsión de las 2 primeras imágenes de la carpeta ``fisheye``

In [ ]:
# TODO Search in the documentation to define 'dim'
# 'dim' is the size (width, height) of the undistorted output image: we keep the original size
dim = gray_img.shape[::-1]
map1, map2 = cv2.fisheye.initUndistortRectifyMap(intrinsics, distortion, np.eye(3), intrinsics, dim, cv2.CV_16SC2)

In [ ]:
# TODO Homework: correct distortion using cv2.remap()
import matplotlib.pyplot as plt

print("Intrinsics:")
print(intrinsics)
print("Distortion coefficients:")
print(distortion)

output_dir = os.path.join("..", "results")
os.makedirs(output_dir, exist_ok=True)

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for i, (img, path) in enumerate(zip(fisheye_imgs[:2], fisheye_imgs_path[:2])):
    undistorted_img = cv2.remap(img, map1, map2, interpolation=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT)
    name = os.path.splitext(os.path.basename(path))[0]
    cv2.imwrite(os.path.join(output_dir, f"{name}_undistorted.jpg"), undistorted_img)

    axes[i, 0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[i, 0].set_title(f"{name} (original)")
    axes[i, 1].imshow(cv2.cvtColor(undistorted_img, cv2.COLOR_BGR2RGB))
    axes[i, 1].set_title(f"{name} (undistorted)")
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()